# LOADING DATA

In [49]:
import pandas as pd
import pickle
from pathlib import Path

# Get the Code directory (project root)
current_dir = Path.cwd()  # from_scratch directory
code_dir = current_dir.parent.parent.parent.parent  # Go up to Code directory
print(f"Code directory: {code_dir}")

# Define all data paths directly
PATHS = {
    # Training features
    'X_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote.parquet',
    'X_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_tomek.parquet',
    'X_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Normalized_Data_split' / 'X_train_smote_tomek.parquet',
    
    # Training targets
    'y_train_smote': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote' / 'y_smote.pkl',
    'y_train_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'tomek' / 'y_tomek.pkl',
    'y_train_smote_tomek': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'smote_tomek' / 'y_smote_tomek.pkl',
    
    # Validation and test features
    'X_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'X_val.parquet',
    'X_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'X_test.parquet',
    
    # Validation and test targets
    'y_val': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'val' / 'y_val.pkl',
    'y_test': code_dir / 'Merged_Data_Preprocessing' / 'Resampled_Data_split' / 'test' / 'y_test.pkl',
}

# Print all paths for verification
print("\nData paths:")
for key, path in PATHS.items():
    exists = "✓" if path.exists() else "✗"
    print(f"  {exists} {key}: {path}")

# Load all data
def load_all_data():
    """Load all data files"""
    data = {}
    
    print("\n" + "="*50)
    print("LOADING DATA")
    print("="*50)
    
    # Load parquet files
    parquet_keys = ['X_train_smote', 'X_train_tomek', 'X_train_smote_tomek', 'X_val', 'X_test']
    for key in parquet_keys:
        path = PATHS[key]
        if path.exists():
            try:
                data[key] = pd.read_parquet(path)
                print(f"✓ Loaded {key}: {data[key].shape}")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
        else:
            print(f"✗ Skipping {key}: file not found at {path}")
    
    # Load pickle files
    pickle_keys = ['y_train_smote', 'y_train_tomek', 'y_train_smote_tomek', 'y_val', 'y_test']
    for key in pickle_keys:
        path = PATHS[key]
        if path.exists():
            try:
                with open(path, 'rb') as f:
                    data[key] = pickle.load(f)
                print(f"✓ Loaded {key}: {len(data[key])} samples")
            except Exception as e:
                print(f"✗ Error loading {key}: {e}")
        else:
            print(f"✗ Skipping {key}: file not found at {path}")
    
    return data

# Load the data
data = load_all_data()

if data:
    print(f"\n" + "="*50)
    print(f"Successfully loaded {len(data)} datasets")
    print("="*50)
    for key, value in data.items():
        if hasattr(value, 'shape'):
            print(f"  {key}: {value.shape}")
        else:
            print(f"  {key}: {len(value)} samples")
else:
    print("\nNo data was loaded")

Code directory: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code

Data paths:
  ✓ X_train_smote: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_smote.parquet
  ✓ X_train_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_tomek.parquet
  ✓ X_train_smote_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Normalized_Data_split\X_train_smote_tomek.parquet
  ✓ y_train_smote: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Resampled_Data_split\smote\y_smote.pkl
  ✓ y_train_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessing\Resampled_Data_split\tomek\y_tomek.pkl
  ✓ y_train_smote_tomek: c:\Users\moous\Documents\M2\Fire-Detection-Data-Mining-Project\Code\Merged_Data_Preprocessi

# DT

In [52]:
import numpy as np
from collections import Counter
import pandas as pd

class DecisionTreeScratch:
    """
    Decision Tree Classifier implemented from scratch
    
    Parameters:
    -----------
    max_depth : int, default=None
        Maximum depth of the tree
    min_samples_split : int, default=2
        Minimum number of samples required to split a node
    min_samples_leaf : int, default=1
        Minimum number of samples required to be at a leaf node
    criterion : str, default='gini'
        Splitting criterion: 'gini' or 'entropy'
    max_features : int or str, default=None
        Number of features to consider for best split
    """
    
    def __init__(self, max_depth=None, min_samples_split=2, 
                 min_samples_leaf=1, criterion='gini', max_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.criterion = criterion
        self.max_features = max_features
        self.tree = None
        self.feature_names = None
        
    def _calculate_impurity(self, y):
        """Calculate impurity (Gini or Entropy)"""
        _, counts = np.unique(y, return_counts=True)
        probabilities = counts / counts.sum()
        
        if self.criterion == 'gini':
            return 1 - np.sum(probabilities ** 2)
        else:  # entropy
            return -np.sum(probabilities * np.log2(probabilities + 1e-10))
    
    def _information_gain(self, parent, left_child, right_child):
        """Calculate information gain for a split"""
        n = len(parent)
        n_left, n_right = len(left_child), len(right_child)
        
        # Weighted average of child impurities
        child_impurity = (n_left/n) * self._calculate_impurity(left_child) + \
                         (n_right/n) * self._calculate_impurity(right_child)
        
        return self._calculate_impurity(parent) - child_impurity
    
    def _best_split(self, X, y):
        """Find the best split for a node"""
        best_gain = -1
        best_feature = None
        best_threshold = None
        
        n_samples, n_features = X.shape
        
        # Determine number of features to consider
        if self.max_features:
            if self.max_features == 'sqrt':
                n_considered = int(np.sqrt(n_features))
            elif self.max_features == 'log2':
                n_considered = int(np.log2(n_features))
            else:
                n_considered = min(self.max_features, n_features)
            feature_indices = np.random.choice(n_features, n_considered, replace=False)
        else:
            feature_indices = range(n_features)
        
        for feature_idx in feature_indices:
            # Get unique values for this feature
            thresholds = np.unique(X[:, feature_idx])
            
            for threshold in thresholds:
                # Create masks for left and right splits
                left_mask = X[:, feature_idx] <= threshold
                right_mask = X[:, feature_idx] > threshold
                
                # Skip if split doesn't meet minimum samples requirements
                if np.sum(left_mask) < self.min_samples_leaf or \
                   np.sum(right_mask) < self.min_samples_leaf:
                    continue
                
                # Calculate information gain
                gain = self._information_gain(y, y[left_mask], y[right_mask])
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature_idx
                    best_threshold = threshold
        
        return best_feature, best_threshold, best_gain
    
    def _create_leaf_node(self, y):
        """Create a leaf node with the majority class"""
        majority_class = Counter(y).most_common(1)[0][0]
        return {'type': 'leaf', 'class': majority_class, 'samples': len(y)}
    
    def _build_tree(self, X, y, depth=0):
        """Recursively build the decision tree"""
        n_samples = X.shape[0]
        
        # Stopping criteria
        if (self.max_depth is not None and depth >= self.max_depth) or \
           n_samples < self.min_samples_split or \
           len(np.unique(y)) == 1:
            return self._create_leaf_node(y)
        
        # Find best split
        feature, threshold, gain = self._best_split(X, y)
        
        # If no good split found
        if gain <= 0:
            return self._create_leaf_node(y)
        
        # Split the data
        left_mask = X[:, feature] <= threshold
        right_mask = X[:, feature] > threshold
        
        # Check if splits meet minimum samples requirement
        if np.sum(left_mask) < self.min_samples_leaf or \
           np.sum(right_mask) < self.min_samples_leaf:
            return self._create_leaf_node(y)
        
        # Recursively build subtrees
        left_subtree = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_subtree = self._build_tree(X[right_mask], y[right_mask], depth + 1)
        
        return {
            'type': 'node',
            'feature': feature,
            'feature_name': self.feature_names[feature] if self.feature_names else f'Feature_{feature}',
            'threshold': threshold,
            'gain': gain,
            'samples': n_samples,
            'depth': depth,
            'left': left_subtree,
            'right': right_subtree
        }
    
    def fit(self, X, y, feature_names=None):
        """
        Fit the decision tree classifier
        
        Parameters:
        -----------
        X : array-like or DataFrame
            Training features
        y : array-like
            Training labels
        feature_names : list, optional
            Names of features for better visualization
        """
        # Convert to numpy arrays
        if isinstance(X, pd.DataFrame):
            X = X.values
            if feature_names is None:
                feature_names = X.columns.tolist() if hasattr(X, 'columns') else None
        
        if isinstance(y, pd.Series) or isinstance(y, pd.DataFrame):
            y = y.values.flatten()
        
        X = np.array(X)
        y = np.array(y)
        
        self.feature_names = feature_names
        self.tree = self._build_tree(X, y)
        
        return self
    
    def _predict_single(self, x, node):
        """Predict a single sample"""
        if node['type'] == 'leaf':
            return node['class']
        
        if x[node['feature']] <= node['threshold']:
            return self._predict_single(x, node['left'])
        else:
            return self._predict_single(x, node['right'])
    
    def predict(self, X):
        """
        Predict class labels for samples in X
        
        Parameters:
        -----------
        X : array-like or DataFrame
            Input samples
            
        Returns:
        --------
        y_pred : array
            Predicted class labels
        """
        if isinstance(X, pd.DataFrame):
            X = X.values
        
        X = np.array(X)
        return np.array([self._predict_single(x, self.tree) for x in X])
    
    def predict_proba(self, X):
        """
        Predict class probabilities for samples in X
        
        Parameters:
        -----------
        X : array-like or DataFrame
            Input samples
            
        Returns:
        --------
        probas : array of shape (n_samples, n_classes)
            Class probabilities
        """
        if isinstance(X, pd.DataFrame):
            X = X.values
        
        X = np.array(X)
        # For now, return hard probabilities (0 or 1)
        # You can enhance this to return soft probabilities
        predictions = self.predict(X)
        n_classes = len(np.unique(self._get_all_labels(self.tree)))
        probas = np.zeros((len(X), n_classes))
        
        for i, pred in enumerate(predictions):
            probas[i, int(pred)] = 1
        
        return probas
    
    def _get_all_labels(self, node):
        """Get all unique labels from the tree"""
        if node['type'] == 'leaf':
            return [node['class']]
        else:
            left_labels = self._get_all_labels(node['left'])
            right_labels = self._get_all_labels(node['right'])
            return list(set(left_labels + right_labels))
    
    def accuracy_score(self, X, y):
        """Calculate accuracy score"""
        y_pred = self.predict(X)
        return np.mean(y_pred == y)
    
    def print_tree(self, node=None, indent="", feature_names=None):
        """Print the decision tree structure"""
        if node is None:
            node = self.tree
            print("\nDecision Tree Structure:")
            print("-" * 50)
        
        if node['type'] == 'leaf':
            print(f"{indent}Leaf: Class {node['class']} (Samples: {node['samples']})")
        else:
            feature_name = node['feature_name']
            print(f"{indent}[Feature {feature_name} <= {node['threshold']:.3f}]")
            print(f"{indent}  ├── True:")
            self.print_tree(node['left'], indent + "  │   ", feature_names)
            print(f"{indent}  └── False:")
            self.print_tree(node['right'], indent + "      ", feature_names)
    
    def get_depth(self, node=None):
        """Get the depth of the tree"""
        if node is None:
            node = self.tree
        
        if node['type'] == 'leaf':
            return 1
        else:
            left_depth = self.get_depth(node['left'])
            right_depth = self.get_depth(node['right'])
            return max(left_depth, right_depth) + 1
    
    def get_n_leaves(self, node=None):
        """Get the number of leaf nodes"""
        if node is None:
            node = self.tree
        
        if node['type'] == 'leaf':
            return 1
        else:
            return self.get_n_leaves(node['left']) + self.get_n_leaves(node['right'])

In [ ]:
# Prepare your datasets
datasets = {
    'smote': {
        'X_train': data['X_train_smote'],
        'y_train': data['y_train_smote'],
        'X_val': data['X_val'],
        'y_val': data['y_val'],
        'X_test': data['X_test'],
        'y_test': data['y_test']
    },
    'tomek': {
        'X_train': data['X_train_tomek'],
        'y_train': data['y_train_tomek'],
        'X_val': data['X_val'],
        'y_val': data['y_val'],
        'X_test': data['X_test'],
        'y_test': data['y_test']
    },
    'smote_tomek': {
        'X_train': data['X_train_smote_tomek'],
        'y_train': data['y_train_smote_tomek'],
        'X_val': data['X_val'],
        'y_val': data['y_val'],
        'X_test': data['X_test'],
        'y_test': data['y_test']
    }
}

# Function to train and evaluate Decision Tree
def train_evaluate_decision_tree(X_train, y_train, X_test, y_test, 
                                 max_depth=5, criterion='gini', 
                                 strategy_name=""):
    """
    Train and evaluate Decision Tree from scratch
    
    Parameters:
    -----------
    X_train, y_train: Training data
    X_test, y_test: Testing data
    max_depth: Maximum tree depth
    criterion: 'gini' or 'entropy'
    strategy_name: Name of resampling strategy for display
    """
    print(f"\n{'='*60}")
    print(f"Training Decision Tree: {strategy_name.upper()}")
    print(f"{'='*60}")
    
    # Initialize and train the tree
    dt = DecisionTreeScratch(max_depth=max_depth, criterion=criterion)
    
    # Get feature names if available
    feature_names = None
    if hasattr(X_train, 'columns'):
        feature_names = X_train.columns.tolist()
    
    print(f"Training samples: {X_train.shape[0]}")
    print(f"Test samples: {X_test.shape[0]}")
    
    # Fit the model
    dt.fit(X_train, y_train, feature_names=feature_names)
    
    # Make predictions
    y_train_pred = dt.predict(X_train)
    y_test_pred = dt.predict(X_test)
    
    # Calculate accuracies
    train_acc = np.mean(y_train_pred == y_train)
    test_acc = np.mean(y_test_pred == y_test)
    
    print(f"\nTraining Accuracy: {train_acc:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")
    
    # Print tree information
    print(f"\nTree Depth: {dt.get_depth()}")
    print(f"Number of Leaves: {dt.get_n_leaves()}")
    
    return dt, train_acc, test_acc

# Train on all three datasets
results = {}
for strategy_name, dataset in datasets.items():
    dt_model, train_acc, test_acc = train_evaluate_decision_tree(
        dataset['X_train'],
        dataset['y_train'],
        dataset['X_test'],
        dataset['y_test'],
        max_depth=5,
        criterion='gini',
        strategy_name=strategy_name
    )
    
    results[strategy_name] = {
        'model': dt_model,
        'train_accuracy': train_acc,
        'test_accuracy': test_acc
    }

# Compare results
print("\n" + "="*60)
print("COMPARISON OF ALL STRATEGIES")
print("="*60)

comparison_data = []
for strategy, metrics in results.items():
    comparison_data.append({
        'Strategy': strategy.upper(),
        'Training Samples': datasets[strategy]['X_train'].shape[0],
        'Train Accuracy': f"{metrics['train_accuracy']:.4f}",
        'Test Accuracy': f"{metrics['test_accuracy']:.4f}"
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n", comparison_df.to_string(index=False))


Training Decision Tree: SMOTE
Training samples: 30570
Test samples: 3477


In [ ]:
import matplotlib.pyplot as plt

def plot_decision_boundaries(dt, X, y, title="Decision Tree Boundaries"):
    """Plot decision boundaries (for 2D data)"""
    if X.shape[1] != 2:
        print("Warning: Can only plot decision boundaries for 2D data")
        return
    
    # Create a mesh grid
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                         np.arange(y_min, y_max, 0.1))
    
    # Predict for each point in the mesh
    Z = dt.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot
    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu')
    plt.scatter(X[:, 0], X[:, 1], c=y, s=20, edgecolor='k', cmap='RdYlBu')
    plt.title(title)
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.show()

# If you want to visualize with PCA (since you have 64 features)
from sklearn.decomposition import PCA

def visualize_with_pca(dt, X, y, title="Decision Tree (PCA Visualization)"):
    """Visualize decision tree predictions using PCA"""
    # Reduce to 2D using PCA
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)
    
    # Create mesh grid in PCA space
    x_min, x_max = X_pca[:, 0].min() - 1, X_pca[:, 0].max() + 1
    y_min, y_max = X_pca[:, 1].min() - 1, X_pca[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                         np.arange(y_min, y_max, 0.1))
    
    # Transform mesh grid back to original space and predict
    mesh_points = np.c_[xx.ravel(), yy.ravel()]
    mesh_original = pca.inverse_transform(mesh_points)
    Z = dt.predict(mesh_original)
    Z = Z.reshape(xx.shape)
    
    # Plot
    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, alpha=0.4, cmap='RdYlBu')
    plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, s=20, edgecolor='k', cmap='RdYlBu')
    plt.title(f"{title}\n(PCA Components 1 & 2)")
    plt.xlabel('PCA Component 1')
    plt.ylabel('PCA Component 2')
    plt.show()